## Colecting the data

In [1]:
!wget https://www.dropbox.com/s/pdhwlpi2yeie0ol/movie-reviews-dataset.zip

--2026-07-18 19:37:57--  https://www.dropbox.com/s/pdhwlpi2yeie0ol/movie-reviews-dataset.zip
Resolving www.dropbox.com (www.dropbox.com)... 162.125.4.18, 2620:100:6022:18::a27d:4212
Connecting to www.dropbox.com (www.dropbox.com)|162.125.4.18|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://www.dropbox.com/scl/fi/4r8fb499vfpyrw44fgftj/movie-reviews-dataset.zip?rlkey=79qfzf6683udd2ehdii38y7wt [following]
--2026-07-18 19:37:57--  https://www.dropbox.com/scl/fi/4r8fb499vfpyrw44fgftj/movie-reviews-dataset.zip?rlkey=79qfzf6683udd2ehdii38y7wt
Reusing existing connection to www.dropbox.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://uc09a65992103c7b598bb47ceaa0.dl.dropboxusercontent.com/cd/0/inline/DEi7GgCtd9Ln078EP6QA2jAEmgEl33wTfp9mrGIUEq-UdiV9f1WMS_Hh4Tzd5C-xunqQ8YQfmebcL1iwuDnS-2bV0GCy4A3Vu9soSN-HS9KaYX_YwsI_1_HqJMoswduZHuQ0ixV1dhs53aShc0c48wpV/file# [following]
--2026-07-18 19:37:58--  https://uc09a65992103c7b598bb47ceaa0

In [2]:
!unzip -q "/content/movie-reviews-dataset.zip"

In [3]:
from tensorflow.keras.preprocessing import text_dataset_from_directory
from tensorflow.strings import regex_replace
from tensorflow.keras.layers import TextVectorization
from tensorflow.keras.models import Sequential
from tensorflow.keras import Input
from tensorflow.keras.layers import Dense, GRU, Embedding, Dropout

In [4]:
def prepareData(dir):
  data = text_dataset_from_directory(dir)
  return data.map(
    lambda text, label: (regex_replace(text, '<br />', ' '), label),
  )

In [5]:
train_data = prepareData('movie-reviews-dataset/train')
test_data = prepareData('movie-reviews-dataset/test')

for text_batch, label_batch in train_data.take(1):
  print(text_batch.numpy()[0])
  print(label_batch.numpy()[0])

Found 25000 files belonging to 2 classes.
Found 25000 files belonging to 2 classes.
b'Steve Martin should quit trying to do remakes of classic comedy. He absolutely does not fit this part. Like the woeful remake of the Out Of Towners, this movie falls flat on it\'s face. How anybody ever thought Steve Martin could even come close to Jack Lemmon\'s wonderful performance is beyond me and the same is true for this movie. Dan Ackroyd could have played the Bilko part better. Martin is great when doing his own original characters but fails miserably trying to recreate other people\'s classic work. It\'s a sad statement when the funniest part of a movie is contained in the first line of the credits when the movie is over. The line "The producers gratefully acknowledge the total lack of cooperation by the United States Army" was just about the only line that actually made me laugh. If you want to see the real Bilko, get hold of the original episodes of the Phil Silvers Show. Those are guarante

# Building the Model

In [6]:
model = Sequential()

In [7]:
model.add(Input(shape=(), dtype="string"))

In [8]:
max_tokens = 1000
max_len = 100
vectorize_layer = TextVectorization(
  max_tokens=max_tokens,
  output_mode="int",
  output_sequence_length=max_len,
)

In [9]:
train_texts = train_data.map(lambda text, label: text)
vectorize_layer.adapt(train_texts)

model.add(vectorize_layer)

In [10]:
model.add(Embedding(max_tokens + 1, 128))

model.add(GRU(64))
model.add(Dense(64, activation="relu"))
model.add(Dense(1, activation="sigmoid"))

In [11]:
model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])

In [12]:
model.fit(train_data, epochs=10)

Epoch 1/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 14s 13ms/step - accuracy: 0.6488 - loss: 0.6169
Epoch 2/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 9s 12ms/step - accuracy: 0.7876 - loss: 0.4527
Epoch 3/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 9s 12ms/step - accuracy: 0.8164 - loss: 0.3988
Epoch 4/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 10s 12ms/step - accuracy: 0.8302 - loss: 0.3752
Epoch 5/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 9s 12ms/step - accuracy: 0.8413 - loss: 0.3560
Epoch 6/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.8516 - loss: 0.3342
Epoch 7/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 11s 12ms/step - accuracy: 0.8622 - loss: 0.3120
Epoch 8/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 9s 12ms/step - accuracy: 0.8746 - loss: 0.2886
Epoch 9/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 9s 12ms/step - accuracy: 0.8853 - loss: 0.2648
Epoch 10/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.8980 - loss: 0.2388


In [13]:
model.evaluate(test_data)

782/782 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - accuracy: 0.7762 - loss: 0.6326


[0.6325684189796448, 0.776199996471405]

In [43]:
text = "good movie !"

In [44]:
import tensorflow as tf
model.predict(tf.constant([text]))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step


array([[0.563732]], dtype=float32)